# Stage 05: supervised models

Compares the models that predict species, on the training set only:

| Model | What it sees |
|---|---|
| Chance | nothing: a uniformly random guess |
| Majority class | nothing: always the most common training species |
| Sample rate only | the native sample rate of the recording, nothing about the sound |
| Logistic regression (whole cuts) | mean and standard deviation of each MFCC and delta over the whole cut |
| Logistic regression (windows) | the same statistics over each 3 s window; a cut's windows are averaged |
| LSTM (shuffled frames) | the frames of each 3 s window, in random order |
| LSTM | the frames of each 3 s window, in order |

The two logistic regressions ignore order by construction. The shuffled LSTM is the same model as
the LSTM with the order of the frames destroyed, so the gap between the two measures what order
adds. Each frame's deltas still describe the change over about 40 ms around it, so "order" here
means order beyond that scale.

The LSTM and the shuffled LSTM are trained on the same five seeds, because the seed alone moved
the best macro-F1 by 0.017, more than the gap being measured. Their scores are the mean over the
seeds, with the spread across them.

The test set is used only when `evaluation.evaluate_test` is true in `config.yaml`.

Reads `data/interim/metadata.csv` and `data/features/`. Writes the tables `cv_results` and
`cv_seed_gaps` in `results/tables/`, and `cv_predictions.csv`, `lstm_learning_curves.csv` and
`chosen_settings.json` in `results/outputs/`; once the test set is unlocked, also the tables
`test_results` and `test_gaps` and `results/outputs/test_predictions.csv`.
Runs in about 5 minutes with the test set locked, on an AMD Ryzen 7 9800X3D CPU (8 cores, 16
threads), nearly all of it the LSTM cross-validation.

In [ ]:
# Generated with Claude Code (Anthropic): stage 05, train and compare the supervised models.
import json
import time

import numpy as np
import pandas as pd
from IPython.display import display

from ecoacoustics.config import load_config, resolve_path
from ecoacoustics.evaluation import (
    bootstrap_gap,
    chance_scores,
    gap_spread,
    gaps_by_run,
    macro_f1,
    majority_predictions,
    scores,
    scores_across_runs,
)
from ecoacoustics.features import (
    cut_windows,
    load_clip_stats,
    load_frames,
    seconds_to_frames,
    window_statistics,
)
from ecoacoustics.io import read_metadata
from ecoacoustics.models import (
    fit_predict_logistic,
    fit_predict_rate_tree,
    fit_predict_window_logistic,
    lstm_seeds,
    parallel_jobs,
    train_lstms,
)
from ecoacoustics.plotting import save_table
from ecoacoustics.splits import filter_species
from ecoacoustics.utils import set_seeds

config = load_config()
SEED = config["seed"]
set_seeds(SEED)
FEATURES, LSTM, EVALUATION = config["features"], config["lstm"], config["evaluation"]
C_GRID, MAX_ITER = config["baseline"]["c_grid"], config["baseline"]["max_iter"]
LSTM_SEEDS = lstm_seeds(SEED, LSTM["n_seeds"])
WINDOW = seconds_to_frames(LSTM["window_seconds"], FEATURES)
HOP = seconds_to_frames(LSTM["window_hop_seconds"], FEATURES)
TABLES_DIR = resolve_path(config, "tables_dir")
OUTPUTS_DIR = resolve_path(config, "outputs_dir")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
SCORE_COLUMNS = {
    "macro_f1": "Macro-F1",
    "macro_f1_sd": "Macro-F1 SD",
    "macro_f1_min": "Macro-F1 lowest",
    "macro_f1_max": "Macro-F1 highest",
    "accuracy": "Accuracy",
    "accuracy_sd": "Accuracy SD",
}

## Data

In [ ]:
meta = read_metadata(resolve_path(config, "interim_dir") / "metadata.csv")
features_dir = resolve_path(config, "features_dir")
clip_stats = load_clip_stats(features_dir, FEATURES).set_index("cut_id").loc[meta["cut_id"]]
frames_by_cut = load_frames(features_dir, FEATURES)

# Every array below has one entry per row of metadata.csv, in the same order.
SPECIES = sorted(meta["species"].unique())
N_CLASSES = len(SPECIES)
if unknown := set(EVALUATION["doubtful_species"]) - set(SPECIES):
    raise ValueError(f"evaluation.doubtful_species names unknown species: {sorted(unknown)}")
y = meta["species"].map({name: k for k, name in enumerate(SPECIES)}).to_numpy()
tapes = meta["tape_id"].to_numpy()
rates = meta["sample_rate"].to_numpy()
x_clip = clip_stats.to_numpy()
frames = [frames_by_cut[cut_id] for cut_id in meta["cut_id"]]
x_windows = [window_statistics(cut_windows(f, WINDOW, HOP)) for f in frames]

train = np.flatnonzero(meta["split"] == "train")
test = np.flatnonzero(meta["split"] == "test")
fold_of = meta["cv_fold"].to_numpy()
y_train = y[train]
print(
    f"{len(train)} training cuts on {len(np.unique(tapes[train]))} tapes in "
    f"{len(np.unique(fold_of[train]))} folds; {len(test)} test cuts; {N_CLASSES} species"
)
print(f"Windows of {WINDOW} frames every {HOP} frames: {sum(len(w) for w in x_windows)} in all")

## Cross-validation

The training cuts are already split into 5 folds of whole tapes (stage 02). Each fold is held out
in turn while a fresh model is trained on the other four, so every training cut gets exactly one
prediction from a model that never heard its tape. Scores are computed on these out-of-fold
predictions pooled over all training cuts, so that all 23 species are scored together.

Each model gets one tuned setting: `C`, the inverse strength of the penalty, for the logistic
regressions, and the number of epochs for the LSTMs. The value with the best pooled macro-F1 is
kept. Choosing it on the same predictions makes the cross-validated scores slightly optimistic,
and the LSTMs had more selection opportunity: 40 epochs, after a pilot on validation fold 0
(stage 05b), against 4 values of `C`. The test set gives the final scores.

In [ ]:
def fold_rows(fold):
    """The training rows outside validation fold ``fold``, and the rows inside it."""
    in_fold = fold_of[train] == fold
    return train[~in_fold], train[in_fold]


def cross_validate(fit_predict):
    """Out-of-fold probabilities for every training cut.

    ``fit_predict(fit_rows, eval_rows)`` trains a fresh model on ``fit_rows`` and returns
    probabilities for ``eval_rows``, of shape (n_eval, n_classes), or of shape
    (n_settings, n_eval, n_classes) when it tries several settings at once.
    """
    oof = None
    for fold in np.unique(fold_of[train]):
        fit_rows, eval_rows = fold_rows(fold)
        probs = fit_predict(fit_rows, eval_rows)
        if oof is None:
            oof = np.zeros((*probs.shape[:-2], len(train), N_CLASSES))
        oof[..., np.searchsorted(train, eval_rows), :] = probs
    return oof


def best_setting(oof_by_setting):
    """Index of the setting with the best pooled macro-F1, and the macro-F1 of every setting."""
    f1 = [scores(y_train, probs.argmax(axis=1))["macro_f1"] for probs in oof_by_setting]
    return int(np.argmax(f1)), f1


# For each model, predictions with one row per run: one run per seed for the LSTMs, else one.
cv_rows, cv_predictions, chosen = [], {}, {}


def record(model, setting, oof):
    """Add a model's pooled out-of-fold scores and predictions to the results."""
    cv_predictions[model] = oof.argmax(axis=1)[None]
    row = scores_across_runs(y_train, cv_predictions[model])
    cv_rows.append({"Model": model, "Setting": setting, **row})


def named_runs(predictions):
    """Species names to save: one column per model, or per seed for a model with several runs."""
    columns = {}
    for model, runs in predictions.items():
        if len(runs) == 1:
            columns[model] = [SPECIES[k] for k in runs[0]]
        else:
            for seed, run in zip(LSTM_SEEDS, runs, strict=True):
                columns[f"{model} (seed {seed})"] = [SPECIES[k] for k in run]
    return columns

### Reference models

In [ ]:
rng = np.random.default_rng(SEED)
chance = chance_scores(y_train, np.arange(N_CLASSES), EVALUATION["n_chance_draws"], rng)
cv_rows.append({"Model": "Chance", "Setting": "uniform guess", **chance})

majority = cross_validate(
    lambda fit, ev: np.eye(N_CLASSES)[majority_predictions(y[fit], len(ev))]
)
record("Majority class", "", majority)

rate_tree = cross_validate(
    lambda fit, ev: fit_predict_rate_tree(rates[fit], y[fit], rates[ev], N_CLASSES, SEED)
)
record("Sample rate only", "decision tree", rate_tree)
pd.DataFrame(cv_rows)

### Logistic regression

In [ ]:
lr_clip = cross_validate(
    lambda fit, ev: np.stack(
        [
            fit_predict_logistic(x_clip[fit], y[fit], x_clip[ev], N_CLASSES, c, MAX_ITER)
            for c in C_GRID
        ]
    )
)
lr_windows = cross_validate(
    lambda fit, ev: np.stack(
        [
            fit_predict_window_logistic(
                [x_windows[i] for i in fit],
                y[fit],
                [x_windows[i] for i in ev],
                N_CLASSES,
                c,
                MAX_ITER,
            )
            for c in C_GRID
        ]
    )
)
for name, oof in [
    ("Logistic regression (whole cuts)", lr_clip),
    ("Logistic regression (windows)", lr_windows),
]:
    best, f1 = best_setting(oof)
    chosen[name] = {"C": C_GRID[best]}
    by_c = {c: round(v, 3) for c, v in zip(C_GRID, f1, strict=True)}
    print(name, "- pooled macro-F1 by C:", by_c)
    record(name, f"C = {C_GRID[best]}", oof[best])

### LSTM and its shuffled-order control

Both models are trained on the same five seeds, derived from the seed in `config.yaml`, on a fixed
number of CPU threads (`lstm.num_threads`). Every model, seed and fold is a separate training, so
several run at the same time, each in its own process (`lstm.n_jobs`); this changes only the speed,
not the results. For a given seed they start from the same weights and see the same windows
in the same batches; the only difference is the order of the frames. The gap between them can
therefore be compared seed by seed.

Each model chooses its number of epochs on its own validation curve, averaged over the seeds. Its
score is the mean over the seeds at that epoch, with the spread across them: the standard
deviation, the lowest and the highest. This cell takes nearly all of the
stage's 5 minutes.

In [ ]:
# The two LSTMs, and whether each one shuffles the frames of every window.
LSTM_MODELS = {"LSTM (shuffled frames)": True, "LSTM": False}


def lstm_call(name, seed, fit_rows, eval_rows, n_epochs):
    """The arguments of one LSTM training: train on ``fit_rows``, then predict ``eval_rows``."""
    return {
        "train_frames": [frames[i] for i in fit_rows],
        "y_train": y[fit_rows],
        "eval_frames": [frames[i] for i in eval_rows],
        "n_classes": N_CLASSES,
        "settings": LSTM,
        "window": WINDOW,
        "hop": HOP,
        "n_epochs": n_epochs,
        "shuffle_order": LSTM_MODELS[name],
        "seed": seed,
    }


runs_to_train = [
    (name, seed, fold)
    for name in LSTM_MODELS
    for seed in LSTM_SEEDS
    for fold in np.unique(fold_of[train])
]
start = time.perf_counter()
histories = train_lstms(
    [
        lstm_call(name, seed, *fold_rows(fold), LSTM["max_epochs"])
        for name, seed, fold in runs_to_train
    ],
    LSTM,
)
history_of = dict(zip(runs_to_train, histories, strict=True))
print(
    f"Trained {len(runs_to_train)} LSTMs, {parallel_jobs(LSTM)} at a time, "
    f"in {time.perf_counter() - start:.0f} s"
)


def lstm_cross_validate(name, seed):
    """Predictions of one seed after every epoch, of shape (n_epochs, n_training_cuts)."""
    # cross_validate hands over the rows of each validation fold; the LSTM trained without that
    # fold has already predicted them.
    return cross_validate(lambda fit, ev: history_of[(name, seed, fold_of[ev[0]])]).argmax(axis=2)


curve_rows = []
for name in LSTM_MODELS:
    runs = np.stack([lstm_cross_validate(name, seed) for seed in LSTM_SEEDS])
    curves = [
        {"model": name, "seed": seed, "epoch": epoch + 1, **scores(y_train, predictions)}
        for seed, run in zip(LSTM_SEEDS, runs, strict=True)
        for epoch, predictions in enumerate(run)
    ]
    curve_rows += curves
    average_curve = pd.DataFrame(curves).groupby("epoch")["macro_f1"].mean()
    epochs = int(average_curve.idxmax())
    chosen[name] = {"epochs": epochs, "seeds": LSTM_SEEDS}
    cv_predictions[name] = runs[:, epochs - 1]
    row = scores_across_runs(y_train, cv_predictions[name])
    cv_rows.append({"Model": name, "Setting": f"{epochs} epochs, {len(LSTM_SEEDS)} seeds", **row})
    print(
        f"{name}: mean pooled macro-F1 {row['macro_f1']:.3f}, sd {row['macro_f1_sd']:.3f}, "
        f"after {epochs} epochs"
    )

### Results

In [ ]:
cv_table = pd.DataFrame(cv_rows).reindex(columns=["Model", "Setting", *SCORE_COLUMNS])
cv_table = cv_table.rename(columns=SCORE_COLUMNS)
save_table(cv_table, "cv_results", TABLES_DIR, "%.3f")

# The gaps seed by seed, with each LSTM at its own chosen number of epochs.
lstm_runs = cv_predictions["LSTM"]
seed_table = pd.DataFrame(
    {
        "Seed": [str(seed) for seed in LSTM_SEEDS],
        "LSTM": [macro_f1(y_train, run) for run in lstm_runs],
        "LSTM (shuffled frames)": [
            macro_f1(y_train, run) for run in cv_predictions["LSTM (shuffled frames)"]
        ],
        "LSTM minus shuffled": gaps_by_run(
            y_train, lstm_runs, cv_predictions["LSTM (shuffled frames)"]
        ),
        "LSTM minus logistic regression (whole cuts)": gaps_by_run(
            y_train, lstm_runs, cv_predictions["Logistic regression (whole cuts)"]
        ),
        "LSTM minus logistic regression (windows)": gaps_by_run(
            y_train, lstm_runs, cv_predictions["Logistic regression (windows)"]
        ),
    }
)
seed_summary = seed_table.drop(columns="Seed").agg(["mean", "std", "min", "max"])
seed_summary.insert(0, "Seed", ["Mean", "SD", "Lowest", "Highest"])
seed_table = pd.concat([seed_table, seed_summary], ignore_index=True)
save_table(seed_table, "cv_seed_gaps", TABLES_DIR, "%.3f")

pd.DataFrame(curve_rows).to_csv(OUTPUTS_DIR / "lstm_learning_curves.csv", index=False)
pd.DataFrame(
    {
        "cut_id": meta["cut_id"].to_numpy()[train],
        "tape_id": tapes[train],
        "species": meta["species"].to_numpy()[train],
        **named_runs(cv_predictions),
    }
).to_csv(OUTPUTS_DIR / "cv_predictions.csv", index=False)
(OUTPUTS_DIR / "chosen_settings.json").write_text(json.dumps(chosen, indent=2), encoding="utf-8")
display(seed_table)
cv_table

## Test set

Every model is trained on all training cuts with the settings chosen above, the LSTMs once per
seed, then scored on the test set. An LSTM's score is the mean over its seeds, with the spread
across them. Three gaps in macro-F1 get a bootstrap interval that resamples whole test tapes, and
the spread of the paired gap across seeds:

- LSTM minus logistic regression on the same windows: order and model together, same input.
- LSTM minus logistic regression on whole cuts: hypothesis H1 as stated in the proposal.
- LSTM minus LSTM on shuffled frames: what order adds on its own.

Every score and gap is reported with all species, and again with the cuts of Bearded Seal and
Long-Finned Pilot Whale left out of the evaluation; the models still train on these species. All
the test cuts of these two species are on tapes where other cuts also carried a second species
label, so these tapes may hold both species and their labels are doubtful.

The comparison is then repeated with only the cuts recorded at 16 kHz or above, keeping the
species that still pass the stage 02 thresholds, to check that the results do not rest on the
empty high frequencies of the low-rate recordings.

In [ ]:
COMPARISONS = [
    ("LSTM", "Logistic regression (windows)", "Order and model, same windows"),
    ("LSTM", "Logistic regression (whole cuts)", "H1 as stated"),
    ("LSTM", "LSTM (shuffled frames)", "Order alone"),
]


def final_predictions(fit_rows, eval_rows):
    """Train every model on ``fit_rows`` with the chosen settings and predict ``eval_rows``.

    Each model gets predictions with one row per run: one per seed for the LSTMs, else one.
    """
    clip_c = chosen["Logistic regression (whole cuts)"]["C"]
    window_c = chosen["Logistic regression (windows)"]["C"]
    predictions = {
        "Majority class": majority_predictions(y[fit_rows], len(eval_rows))[None],
        "Sample rate only": fit_predict_rate_tree(
            rates[fit_rows], y[fit_rows], rates[eval_rows], N_CLASSES, SEED
        ).argmax(axis=1)[None],
        "Logistic regression (whole cuts)": fit_predict_logistic(
            x_clip[fit_rows], y[fit_rows], x_clip[eval_rows], N_CLASSES, clip_c, MAX_ITER
        ).argmax(axis=1)[None],
        "Logistic regression (windows)": fit_predict_window_logistic(
            [x_windows[i] for i in fit_rows],
            y[fit_rows],
            [x_windows[i] for i in eval_rows],
            N_CLASSES,
            window_c,
            MAX_ITER,
        ).argmax(axis=1)[None],
    }
    final_runs = [(name, seed) for name in LSTM_MODELS for seed in LSTM_SEEDS]
    final_histories = train_lstms(
        [
            lstm_call(name, seed, fit_rows, eval_rows, chosen[name]["epochs"])
            for name, seed in final_runs
        ],
        LSTM,
    )
    final_history_of = dict(zip(final_runs, final_histories, strict=True))
    for name in LSTM_MODELS:
        predictions[name] = np.stack(
            [final_history_of[(name, seed)][-1].argmax(axis=1) for seed in LSTM_SEEDS]
        )
    return predictions

In [ ]:
if not EVALUATION["evaluate_test"]:
    print(
        "The test set is locked. Set evaluation.evaluate_test to true in config.yaml once the "
        "settings above are final; the test set is then used once."
    )
else:
    prep = config["prepare"]
    min_rate = EVALUATION["robustness_min_sample_rate"]
    kept, _ = filter_species(
        meta[meta["sample_rate"] >= min_rate],
        prep["grouping_key"],
        prep["min_cuts_per_species"],
        prep["min_groups_per_species"],
    )
    robust = meta.index.isin(kept.index)
    data_sets = {
        "All cuts": (train, test),
        f"Cuts at {min_rate} Hz or more": (train[robust[train]], test[robust[test]]),
    }
    species_of_cut = meta["species"].to_numpy()
    doubtful_names = [name.replace("_", " ") for name in EVALUATION["doubtful_species"]]
    without_doubtful = "Without " + " and ".join(doubtful_names)
    rng = np.random.default_rng(SEED)
    test_rows, gap_rows, saved = [], [], []
    for data_name, (fit_rows, eval_rows) in data_sets.items():
        predictions = final_predictions(fit_rows, eval_rows)
        doubtful = np.isin(species_of_cut[eval_rows], EVALUATION["doubtful_species"])
        all_cuts = np.ones(len(eval_rows), dtype=bool)
        for species_set, keep in [("All species", all_cuts), (without_doubtful, ~doubtful)]:
            labels = {"Data": data_name, "Species": species_set}
            y_eval = y[eval_rows][keep]
            tapes_eval = tapes[eval_rows][keep]
            chance = chance_scores(
                y_eval, np.unique(y[fit_rows]), EVALUATION["n_chance_draws"], rng
            )
            test_rows.append({**labels, "Model": "Chance", **chance})
            for model, runs in predictions.items():
                row = scores_across_runs(y_eval, runs[:, keep])
                test_rows.append({**labels, "Model": model, **row})
            for model_a, model_b, question in COMPARISONS:
                runs_a, runs_b = predictions[model_a][:, keep], predictions[model_b][:, keep]
                gap = bootstrap_gap(
                    y_eval,
                    runs_a,
                    runs_b,
                    tapes_eval,
                    EVALUATION["n_bootstrap"],
                    EVALUATION["ci_level"],
                    rng,
                )
                gap_rows.append(
                    {
                        **labels,
                        "Comparison": f"{model_a} minus {model_b}",
                        "Question": question,
                        **gap,
                        **gap_spread(y_eval, runs_a, runs_b),
                    }
                )
        saved.append(
            pd.DataFrame(
                {
                    "data": data_name,
                    "cut_id": meta["cut_id"].to_numpy()[eval_rows],
                    "tape_id": tapes[eval_rows],
                    "species": species_of_cut[eval_rows],
                    **named_runs(predictions),
                }
            )
        )
    test_table = pd.DataFrame(test_rows).reindex(
        columns=["Data", "Species", "Model", *SCORE_COLUMNS]
    )
    test_table = test_table.rename(columns=SCORE_COLUMNS)
    gap_table = pd.DataFrame(gap_rows).rename(
        columns={
            "gap": "Macro-F1 gap",
            "ci_low": "CI low",
            "ci_high": "CI high",
            "share_a_better": "Share of resamples with first model better",
            "gap_sd": "Gap SD across seeds",
            "gap_min": "Lowest gap across seeds",
            "gap_max": "Highest gap across seeds",
        }
    )
    save_table(test_table, "test_results", TABLES_DIR, "%.3f")
    save_table(gap_table, "test_gaps", TABLES_DIR, "%.3f")
    pd.concat(saved).to_csv(OUTPUTS_DIR / "test_predictions.csv", index=False)
    display(test_table)
    display(gap_table)